# Sokoban

En esta práctica desarrollaremos un agente capaz de resolver Sokoban mediante búsqueda en espacios de estados. El estado solo describe el tablero; el algoritmo de búsqueda guarda por separado los costes y los estados anteriores.

![Sokoban](sokoban.png)

Un robot se mueve en cuatro direcciones y puede empujar cajas, pero no arrastrarlas. El objetivo es colocar todas las cajas en las zonas de almacenamiento. Puedes probar el juego en [Sokoban Online](http://www.sokobanonline.com/play).

## Ejercicio 1: representación del estado

Representa el estado con una tupla `(robots, caixes)`. `robots` es una tupla de posiciones y `caixes` es un `frozenset` de posiciones.

Las dimensiones del tablero, los almacenes y los obstáculos no cambian durante una partida; guárdalos en un diccionario `problema`, separado del estado. No guardes en el estado la acción, el coste ni el estado anterior: son datos del algoritmo de búsqueda.

Trabaja con las listas `exemples_facils`, `exemples_mitjans` y `exemples_dificils`. Cada elemento es una tupla `(nom, problema, inicial)`. Todavía no hace falta resolverlos.

In [ ]:
Posicio = tuple[int, int]

def crea_problema(amplada, altura, magatzems, obstacles=()):
    return {
        'amplada': amplada,
        'altura': altura,
        'magatzems': frozenset(magatzems),
        'obstacles': frozenset(obstacles),
    }

exemples_facils = [
    ('Fàcil 1 — desviament voraç', crea_problema(5, 5, {(1, 4)}, {(1, 0), (0, 2), (1, 2)}), (((2, 4),), frozenset({(1, 3)}))),
    ('Fàcil 2 — empenta horitzontal', crea_problema(5, 5, {(4, 2)}), (((1, 2),), frozenset({(2, 2)}))),
    ('Fàcil 3 — empenta horitzontal inversa', crea_problema(5, 5, {(0, 3)}), (((3, 3),), frozenset({(2, 3)}))),
    ('Fàcil 4 — dos robots amb obstacles', crea_problema(5, 5, {(0, 0)}, {(1, 0), (2, 0), (3, 0), (1, 4), (2, 4), (3, 4)}), (((2, 1), (2, 3)), frozenset({(1, 1)}))),
]

exemples_mitjans = [
    ('Mitjà 1 — desviament voraç amb dues caixes', crea_problema(6, 4, {(2, 1), (2, 2)}), (((2, 1), (2, 2)), frozenset({(1, 1), (4, 2)}))),
    ('Mitjà 2 — obstacle central', crea_problema(6, 5, {(4, 1), (4, 3)}, {(3, 2)}), (((1, 1), (1, 3)), frozenset({(2, 1), (2, 3)}))),
    ('Mitjà 3 — caixes en paral·lel', crea_problema(7, 5, {(1, 0), (5, 4)}), (((1, 3), (5, 1)), frozenset({(1, 2), (5, 2)}))),
]

exemples_dificils = [
    ('Difícil 1 — zona oberta i desempat', crea_problema(8, 8, {(3, 1), (4, 6), (7, 0)}, {(0, 1), (1, 5), (2, 3), (3, 3), (5, 0)}), (((0, 6),), frozenset({(6, 3), (2, 5), (3, 6)}))),
    ('Difícil 2 — tres caixes verticals', crea_problema(10, 9, {(9, 0), (9, 4), (9, 8)}), (((0, 0), (0, 4), (0, 8)), frozenset({(1, 0), (1, 4), (1, 8)}))),
    ('Difícil 3 — tres caixes en tauler ample', crea_problema(12, 10, {(11, 1), (11, 4), (11, 7)}), (((0, 1), (0, 4), (0, 7)), frozenset({(1, 1), (1, 4), (1, 7)}))),
    ('Difícil 4 — tres carrils de 6×6', crea_problema(6, 6, {(5, 0), (5, 2), (0, 5)}), (((0, 0), (0, 2), (0, 4)), frozenset({(1, 0), (1, 2), (1, 4)}))),
    ('Difícil 5 — tres carrils de 7×7', crea_problema(7, 7, {(6, 0), (6, 3), (6, 6)}), (((0, 0), (0, 3), (0, 6)), frozenset({(1, 0), (1, 3), (1, 6)}))),
    ('Difícil 6 — tres carrils de 8×7', crea_problema(8, 7, {(7, 1), (7, 3), (7, 5)}), (((0, 1), (0, 3), (0, 5)), frozenset({(1, 1), (1, 3), (1, 5)}))),
    ('Difícil 7 — tres objectius amb recorregut lateral', crea_problema(8, 8, {(7, 0), (7, 3), (0, 7)}), (((0, 0), (0, 3), (0, 6)), frozenset({(1, 0), (1, 3), (1, 6)}))),
]

problema_facil, inicial_facil = exemples_facils[0][1:]
problema_mitja, inicial_mitja = exemples_mitjans[0][1:]
problema_dificil, inicial_dificil = exemples_dificils[0][1:]


## Ejercicio 2: sucesores

Implementa estas funciones, sin clases de problema:

- `dins_tauler(posicio, problema)`
- `successors(estat, problema)`, que devuelve pares `(nou_estat, cost)`
- `es_objectiu(estat, problema)`
- `heuristica(estat, problema)`, con la distancia Manhattan desde cada caja hasta el almacén más cercano.
- `mostra_estat(estat, problema)`, que representa el tablero con emojis y un marco.

Todos los movimientos cuestan 1. Prueba `successors(inicial_facil, problema_facil)` y comprueba que cada movimiento sea legal: no puede salir del tablero, atravesar obstáculos ni empujar una caja si no hay espacio delante. Ordena siempre la tupla de robots antes de devolver el estado para no representar el mismo tablero de dos maneras distintas.

## Ejercicio 3: UCS

Implementa `ucs(problema, inicial)`. La frontera es un `heapq`; `visitats` evita expandir un mismo estado más de una vez y `pares` permite reconstruir la ruta.

Devuelve `final, cost, ruta, visitats`, o `None, None, None, visitats` si no hay solución. `ruta` contiene los estados desde el inicial hasta el final; `visitats` cuenta los estados extraídos de la frontera.

Resuelve `problema_facil` con UCS y muestra el coste, los estados visitados y cada estado de la ruta.

In [ ]:
final, cost, ruta, visitats = ucs(problema_facil, inicial_facil)
print(cost)


## Ejercicio 4: Greedy y A*

Una heurística es **admisible** si nunca sobreestima el coste que falta para llegar al objetivo. La suma de las distancias Manhattan desde cada caja hasta el almacén más cercano es admisible, porque ignora los obstáculos y las interferencias entre cajas.

Implementa `greedy(problema, inicial)` con prioridad `heuristica(estat)` y `a_star(problema, inicial)` con prioridad `cost + heuristica(estat)`.

En A* utiliza el mismo esquema que en UCS. Compara el coste de las rutas y el número de estados visitados.

Resuelve los ejemplos fáciles con Greedy y A*. Después prueba los ejemplos intermedios: en este punto todavía son suficientemente pequeños para poder comparar ambos algoritmos.

## Ejercicio 5: comparación

Ya tienes cuatro tableros fáciles, tres intermedios y siete difíciles. El primero de cada categoría está diseñado para que Greedy encuentre una ruta no óptima. Compara UCS, Greedy y A*: coste, longitud de la ruta, estados visitados y tiempo de ejecución. Puedes utilizar `time.time()` para medir el tiempo.

¿Qué algoritmo es más eficiente en tiempo y en estados visitados? ¿Cuál encuentra siempre la ruta óptima? ¿Por qué? ¿Qué información aporta la heurística?

Completa la comparación con cada ejemplo difícil usando solo Greedy y A*. UCS puede ser demasiado lento en estos niveles. Utiliza un límite de estados expandidos para evitar que una ejecución larga bloquee el notebook; si se alcanza el límite, indícalo como resultado.

## Ejercicio 6: desempate en A*

Los tableros difíciles pueden ser lentos: en una zona abierta, muchos estados tienen la misma prioridad `f = g + h`, porque la heurística aporta poca información. A* tiene que elegir entre muchos candidatos igual de buenos.

Prueba una estrategia de **desempate**: si dos estados tienen el mismo valor `f`, da prioridad al que tenga un valor `g` mayor. Con `heapq` puedes usar la prioridad `(f, -g, comptador, estat)`. El contador evita que Python intente comparar directamente dos estados.

Implementa `a_star_millorat(problema, inicial)` y compáralo con A* en los tableros difíciles. El desempate no cambia la optimalidad, solo el orden de exploración.

Resuelve los ejemplos difíciles con ambas versiones de A* y compara los estados visitados. En el primer ejemplo difícil, A* sin desempate puede alcanzar el límite antes de encontrar la solución; tenlo en cuenta en la comparación.